# Entity Resolution - Simple Refactored Version

Match organizations from clinical trials against reference organization data.

In [ ]:
# Imports
import os
import gc
import logging
from pathlib import Path
from typing import Dict, List, Set
import pandas as pd
import polars as pl
import duckdb
from pyathena import connect
from pyathena.pandas.cursor import PandasCursor
from splink import DuckDBAPI, Linker

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info("✓ Imports complete")

In [ ]:


def execute_query(query: str, description: str = "") -> pd.DataFrame:
    """Execute query and return dataframe"""
    logger.info(f"🔄 {description}")
    conn = get_db_connection()
    cursor = conn.cursor()
    result = cursor.execute(query).as_pandas()
    conn.close()
    logger.info(f"✓ Loaded {len(result):,} rows")
    return result

In [ ]:
# Helper functions
def parse_metadata(metadata_str: str) -> Dict:
    """Parse metadata string into dictionary"""
    if pd.isna(metadata_str) or metadata_str == '{}':
        return {}
    try:
        metadata_str = str(metadata_str).strip('{}')
        result = {}
        for part in metadata_str.split(', '):
            if '=' in part:
                key, value = part.split('=', 1)
                if key not in ['name_clean', 'name_normalized', 'name_prefix_10', 'name_prefix_5']:
                    if value == 'null':
                        result[key] = None
                    else:
                        result[key] = value
        return result
    except:
        return {}

def standardize_country(country):
    """Standardize country codes"""
    if not country or pd.isna(country):
        return None
    country = str(country).upper().strip()
    # Map common variations
    mapping = {
    'united states': 'US', 'usa': 'US', 'america': 'US', 'us': 'US',
    'united kingdom': 'UK', 'britain': 'UK', 'great britain': 'UK', 'uk': 'UK',
    'germany': 'DE', 'deutschland': 'DE',
    'france': 'FR', 'italy': 'IT', 'spain': 'ES',
    'canada': 'CA', 'australia': 'AU', 'japan': 'JP',
    'china': 'CN', 'india': 'IN', 'brazil': 'BR'
}
    return mapping.get(country, country[:2] if len(country) > 2 else country)

In [ ]:
# Step 1: Load mismatched organizations
query_mismatched = """
SELECT * 
FROM "allsci_prod_gold"."potential_mismatched_organizations"
"""

all_mismatched = execute_query(query_mismatched, "Loading mismatched organizations")

In [ ]:
# Step 2: Filter to organizations in all three sources
def get_orgs_in_all_sources(df: pd.DataFrame) -> pd.DataFrame:
    """Get organizations that appear in all three sources"""
    names_by_source = []
    
    for source in CLINICAL_TRIAL_SOURCES:
        names = set(df[df['source_table'] == source]['name'].unique())
        names_by_source.append(names)
        logger.info(f"  {source.split('.')[-1]}: {len(names):,} orgs")
    
    # Find intersection
    common_orgs = names_by_source[0].intersection(*names_by_source[1:])
    logger.info(f"✓ Found {len(common_orgs):,} organizations in all three sources")
    
    return df[df['name'].isin(common_orgs)]

filtered_orgs = get_orgs_in_all_sources(all_mismatched)

In [ ]:
filtered_orgs

In [ ]:
# Step 3: Create wide-format dataframe
def create_wide_format(df: pd.DataFrame) -> pd.DataFrame:
    """Create one row per organization with columns from all sources"""
    # Parse metadata
    df['metadata_dict'] = df['metadata'].apply(parse_metadata)
    
    # Split by source
    org_df = df[df['source_table'] == CLINICAL_TRIAL_SOURCES[0]].copy()
    sponsor_df = df[df['source_table'] == CLINICAL_TRIAL_SOURCES[1]].copy()
    location_df = df[df['source_table'] == CLINICAL_TRIAL_SOURCES[2]].copy()
    
    # Get unique names
    unique_names = df['name'].unique()
    result = pd.DataFrame({'name': unique_names})
    
    # Add columns from each source
    for source_df, prefix in [(org_df, 'org'), (sponsor_df, 'sponsor'), (location_df, 'location')]:
        if len(source_df) > 0:
            # Group by name and take first record's metadata
            source_data = []
            for name in unique_names:
                name_records = source_df[source_df['name'] == name]
                if len(name_records) > 0:
                    metadata = name_records.iloc[0]['metadata_dict']
                    metadata['name'] = name
                    metadata[f'{prefix}_count'] = len(name_records)
                    source_data.append(metadata)
                else:
                    source_data.append({'name': name, f'{prefix}_count': 0})
            
            source_wide = pd.DataFrame(source_data)
            # Rename columns with prefix
            rename_dict = {col: f'{prefix}_{col}' for col in source_wide.columns if col != 'name'}
            source_wide = source_wide.rename(columns=rename_dict)
            
            result = result.merge(source_wide, on='name', how='left')
    
    # Add unique ID
    result['unique_id'] = range(1, len(result) + 1)
    result['unique_id'] = result['unique_id'].astype(str)
    
    logger.info(f"✓ Created wide format: {len(result):,} organizations")
    return result

wide_df = create_wide_format(filtered_orgs)
print(f"\nShape: {wide_df.shape}")
print(f"Columns: {list(wide_df.columns)[:10]}...")

In [ ]:
wide_df['org_org_count'].sum()

In [ ]:
# Step 4: Prepare data for entity resolution
def prepare_for_matching(df: pd.DataFrame) -> pd.DataFrame:
    """Extract key fields for matching"""
    result = pd.DataFrame({
        'unique_id': df['unique_id'],
        'name': df['name'],
        'city': df.get('location_city'),
        'country': df.get('location_country', pd.Series([None]*len(df))).apply(standardize_country),
        'types': None,
        'grid': None,
        'ror': None
    })
    
    # Clean nulls
    for col in ['city', 'country', 'types', 'grid', 'ror']:
        result[col] = result[col].where(result[col].notna(), None)
    
    return result

mismatched_for_matching = prepare_for_matching(wide_df)
logger.info(f"✓ Prepared {len(mismatched_for_matching):,} organizations for matching")

In [ ]:
mismatched_for_matching

In [ ]:
mismatched_for_matching.to_csv('mismatched_for_matching.csv')

In [ ]:
# Step 5: Load reference data
query_reference = """
SELECT allsci_id, name, country_code, type, geographic_location
FROM "allsci_prod_gold"."dim_organization"
"""

reference_data = execute_query(query_reference, "Loading reference organizations")

In [ ]:
# Step 6: Prepare reference data
def prepare_reference_data(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare reference data for matching"""
    # Parse geographic location
    def parse_geo(geo_str):
        if pd.isna(geo_str):
            return {}
        return parse_metadata(geo_str)
    
    geo_data = df['geographic_location'].apply(parse_geo)
    
    result = pd.DataFrame({
        'unique_id': df['allsci_id'].astype(str),
        'name': df['name'],
        'city': geo_data.apply(lambda x: x.get('city')),
        'country': df['country_code'].apply(standardize_country),
        'types': df['type'],
        'grid': None,
        'ror': None
    })
    
    # Clean nulls
    for col in ['city', 'country', 'types']:
        result[col] = result[col].where(result[col].notna(), None)
    
    return result

reference_for_matching = prepare_reference_data(reference_data)
logger.info(f"✓ Prepared {len(reference_for_matching):,} reference organizations")

In [ ]:
# Step 7: Create full tokenization pipeline
logger.info("🔄 Creating tokenization pipeline...")

# Convert to Polars
mismatched_pl = pl.from_pandas(mismatched_for_matching)
reference_pl = pl.from_pandas(reference_for_matching)

# Register with DuckDB
duckdb.register("mismatched_orgs", mismatched_pl.to_pandas())
duckdb.register("reference_orgs", reference_pl.to_pandas())

# Create combined dataset
sql_combine = """
CREATE OR REPLACE TABLE all_input_data AS
WITH concat_data AS (
    SELECT *, 'mismatched' as source_dataset FROM mismatched_orgs
    UNION ALL
    SELECT *, 'reference' as source_dataset FROM reference_orgs
)
SELECT 
    ROW_NUMBER() OVER () as unique_id_row, 
    *,
    REGEXP_REPLACE(REGEXP_REPLACE(upper(trim(name)), '[^A-Z0-9\\s\\-&]', ' ', 'g'), '\\s+', ' ', 'g') as clean_name
FROM concat_data
"""
duckdb.execute(sql_combine)

# Tokenize names
sql_tokenize = """
CREATE OR REPLACE TABLE unnested AS
SELECT
    unique_id,
    unnest(regexp_split_to_array(clean_name, '\\s+')) as name_token,
    generate_subscripts(regexp_split_to_array(clean_name, '\\s+'), 1) as token_position_in_name,
    source_dataset
FROM all_input_data
WHERE clean_name IS NOT NULL AND length(trim(clean_name)) >= 2
"""
duckdb.execute(sql_tokenize)

# Calculate token frequencies
sql_frequencies = """
CREATE OR REPLACE TABLE token_frequencies AS
WITH token_stats AS (
    SELECT
        name_token as token,
        COUNT(*) as token_count,
        COUNT(DISTINCT unique_id) as unique_orgs
    FROM unnested
    GROUP BY name_token
),
total_stats AS (
    SELECT COUNT(*) as total_tokens, COUNT(DISTINCT unique_id) as total_orgs FROM unnested
)
SELECT
    ts.*,
    ts.token_count::FLOAT / tt.total_tokens as rel_freq,
    LOG(tt.total_orgs::FLOAT / ts.unique_orgs) as idf_weight
FROM token_stats ts
CROSS JOIN total_stats tt
"""
duckdb.execute(sql_frequencies)

# Create enriched token data
sql_enrich = """
CREATE OR REPLACE TABLE input_data_with_tokens AS
WITH tokens_with_freq AS (
    SELECT
        m.unique_id,
        list_transform(
            list_zip(
                array_agg(u.name_token ORDER BY u.token_position_in_name),
                array_agg(COALESCE(tf.rel_freq, 0.0001) ORDER BY u.token_position_in_name),
                array_agg(COALESCE(tf.idf_weight, 1.0) ORDER BY u.token_position_in_name)
            ),
            x -> struct_pack(token := x[1], rel_freq := x[2], idf_weight := x[3])
        ) as name_tokens_with_freq,
        COUNT(*) as token_count
    FROM all_input_data m
    JOIN unnested u ON m.unique_id = u.unique_id
    LEFT JOIN token_frequencies tf ON u.name_token = tf.token
    GROUP BY m.unique_id
)
SELECT m.*, t.name_tokens_with_freq, t.token_count
FROM all_input_data m
LEFT JOIN tokens_with_freq t ON m.unique_id = t.unique_id
"""
duckdb.execute(sql_enrich)

# Generate rare tokens
sql_rare_tokens = """
CREATE OR REPLACE TABLE all_orgs_tokenized AS
WITH tokens_unnested AS (
    SELECT unique_id, unnest(name_tokens_with_freq) as token_info
    FROM input_data_with_tokens
),
rare_tokens AS (
    SELECT
        unique_id,
        array_agg(token_info.token ORDER BY token_info.rel_freq ASC)[:3] as rarest_tokens,
        array_agg(token_info.token ORDER BY length(token_info.token) DESC)[:2] as longest_tokens
    FROM tokens_unnested
    WHERE token_info.rel_freq < 0.01
    GROUP BY unique_id
)
SELECT 
    m.*,
    COALESCE(r.rarest_tokens, ARRAY[]::VARCHAR[]) as rarest_tokens,
    COALESCE(r.longest_tokens, ARRAY[]::VARCHAR[]) as longest_tokens
FROM input_data_with_tokens m
LEFT JOIN rare_tokens r ON m.unique_id = r.unique_id
ORDER BY m.unique_id
"""
duckdb.execute(sql_rare_tokens)

total_records = duckdb.execute("SELECT COUNT(*) FROM all_orgs_tokenized").fetchone()[0]
logger.info(f"✓ Full tokenization complete: {total_records:,} total records")

In [ ]:
# Step 8: Run entity resolution
def run_matching():
    """Run entity resolution between mismatched and reference orgs"""
    
    # Get tokenized data
    mismatched_final = duckdb.sql(
        "SELECT * FROM all_orgs_tokenized WHERE source_dataset = 'mismatched'"
    ).df()
    
    reference_final = duckdb.sql(
        "SELECT * FROM all_orgs_tokenized WHERE source_dataset = 'reference'"
    ).df()
    
    logger.info(f"🔄 Running entity resolution...")
    logger.info(f"  Mismatched: {len(mismatched_final):,} records")
    logger.info(f"  Reference: {len(reference_final):,} records")
    
    # Create linker
    linker = Linker(
        [mismatched_final, reference_final],
        str(MODEL_PATH),
        db_api=DuckDBAPI(),
        input_table_aliases=["mismatched", "reference"]
    )
    
    # Run predictions
    predictions = linker.inference.predict(threshold_match_probability=0.95)
    matches = predictions.as_pandas_dataframe()
    
    logger.info(f"✓ Found {len(matches):,} matches")
    
    if len(matches) > 0:
        high_conf = len(matches[matches['match_probability'] > 0.9])
        med_conf = len(matches[(matches['match_probability'] > 0.7) & (matches['match_probability'] <= 0.9)])
        logger.info(f"  High confidence (>0.9): {high_conf:,}")
        logger.info(f"  Medium confidence (0.7-0.9): {med_conf:,}")
        logger.info(f"  Average probability: {matches['match_probability'].mean():.3f}")
    
    return matches

matches = run_matching()

In [ ]:
matches.columns

In [ ]:
matches[['name_l', 'name_r', 'source_dataset_l', 'source_dataset_r', 'match_probability','city_l', 'city_r','country_l', 'country_r']].sort_values(by='match_probability', ascending=False).head(20)

In [ ]:
# Step 9: Save results
if len(matches) > 0:
    output_file = OUTPUT_DIR / 'entity_matches.csv'
    
    # Select key columns
    output_cols = [
        'unique_id_l', 'name_l', 'city_l', 'country_l',
        'unique_id_r', 'name_r', 'city_r', 'country_r', 
        'match_probability', 'match_weight'
    ]
    
    available_cols = [col for col in output_cols if col in matches.columns]
    matches[available_cols].to_csv(output_file, index=False)
    logger.info(f"💾 Saved to {output_file}")
    
    # Show top matches
    print("\n🏆 Top 10 matches:")
    top_matches = matches.nlargest(10, 'match_probability')[['name_l', 'name_r', 'match_probability']]
    for _, row in top_matches.iterrows():
        print(f"{row['name_l'][:40]:<40} → {row['name_r'][:40]:<40} ({row['match_probability']:.3f})")

In [ ]:
# Cleanup
gc.collect()
logger.info("\n✅ Complete!")